# TypeOracle + KG-Specialized LLM: Full Experiment

**Purpose:** Run the complete DCA-Trie experiment — TypeOracle path filtering followed by
graph-constrained decoding with the KG-specialized LLM — and compare against the
unfiltered GCR baseline.

**Pipeline:**
1. SIR/FNR evaluation (CPU-only) — measure how many paths the oracle prunes
2. Graph-constrained decoding with TypeOracle-filtered trie
3. Graph-constrained decoding with unfiltered trie (baseline)
4. Head-to-head comparison: Hit@1, path reduction, timing

**Reference:** Luo et al. (2025) GCR; DCA-Trie TypeOracle (this thesis)

In [2]:
# Setup & Environment
import os
import sys
import time
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BASE = "/content/drive/MyDrive/GCR_TypeOracle"
    os.makedirs(DRIVE_BASE, exist_ok=True)
except ImportError:
    IN_COLAB = False
    DRIVE_BASE = None
    print("Not on Colab — skipping Drive mount.")

# GPU Detection
import torch

GPU_ARCH_MAP = {
    "A100": "80", "A100-SXM": "80", "A100-PCIE": "80",
    "L4": "89", "T4": "75", "V100": "70",
    "H100": "90", "H200": "90",
    "RTX 4090": "89", "RTX 3090": "86",
}

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_arch = None
    for key, arch in GPU_ARCH_MAP.items():
        if key in gpu_name:
            gpu_arch = arch
            break
    has_a100 = "A100" in gpu_name
    print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB  |  SM: {gpu_arch}")
else:
    gpu_name = "None"
    gpu_mem = 0
    gpu_arch = None
    has_a100 = False
    print("WARNING: No GPU detected.")

# Clone repo if needed
REPO_DIR = "/content/graph-constrained-reasoning" if IN_COLAB else os.getcwd()
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Adjanour/graph-constrained-reasoning-dca {REPO_DIR}
if IN_COLAB:
    os.chdir(REPO_DIR)
sys.path.insert(0, '.')

# Install dependencies
DEPS = [
    "transformers==4.44.2", "accelerate>=0.30.1", "datasets>=2.19.2",
    "marisa-trie>=1.2.0", "networkx>=3.0", "scikit-learn>=1.5.0",
    "tiktoken>=0.7.0", "sentencepiece>=0.2.0", "protobuf>=5.27.1",
]
print("Installing dependencies...")
!pip install -q {' '.join(DEPS)}

# Install flash-attn (optional).
# Not required -- sdpa works on all GPUs. But on A100,
# flash_attention_2 is ~20% faster for beam search.
#
# Strategy:
#   1. Try pre-compiled wheel from GitHub releases (instant)
#   2. Source install with ninja + arch targeting (~10-15 min)
#   3. Fall back to sdpa (no flash-attn)

INSTALL_FLASH_ATTN = False  #@param {type:"boolean"}

flash_attn_installed = False

if INSTALL_FLASH_ATTN and torch.cuda.is_available():
    # Detect Python + CUDA + PyTorch versions for wheel matching.
    py_ver = f"{sys.version_info.major}{sys.version_info.minor}"
    cuda_ver = torch.version.cuda.replace(".", "")
    torch_ver = torch.__version__.split("+")[0]
    torch_major_minor = ".".join(torch_ver.split(".")[:2])

    print(f"Python: {py_ver}  CUDA: {cuda_ver}  PyTorch: {torch_ver}")

    # Step 1: Try pre-compiled wheel.
    wheel_url = (
        f"https://github.com/Dao-AILab/flash-attention/releases/"
        f"download/v2.7.3/flash_attn-2.7.3+cu{cuda_ver}"
        f"torch{torch_major_minor}cxx11abiFALSE-cp{py_ver}"
        f"-cp{py_ver}-linux_x86_64.whl"
    )
    print("Trying pre-compiled wheel...")
    ret = os.system(f"pip install -q '{wheel_url}' 2>/dev/null")
    if ret == 0:
        flash_attn_installed = True
        print("flash-attn installed from pre-compiled wheel.")
    else:
        print("Pre-compiled wheel not available for this combination.")

    # Step 2: Source install with ninja + arch targeting.
    if not flash_attn_installed:
        print("\nFalling back to source install with ninja...")
        !pip install -q ninja

        # Set target architecture to avoid compiling for all GPU types.
        if gpu_arch:
            os.environ["FLASH_ATTN_CUDA_ARCHS"] = gpu_arch
            print(f"Targeting SM {gpu_arch} only (your GPU: {gpu_name})")

        # Ninja parallelizes compilation across all CPU cores.
        n_cores = os.cpu_count() or 1
        print(f"Compiling with ninja ({n_cores} cores), ~10-15 min...")

        t0 = time.time()
        !pip install flash-attn --no-build-isolation
        elapsed = time.time() - t0

        try:
            import flash_attn
            flash_attn_installed = True
            print(
                f"flash-attn installed in {elapsed:.0f}s "
                f"(v{flash_attn.__version__})"
            )
        except ImportError:
            print(
                f"flash-attn install failed after {elapsed:.0f}s. "
                f"Using sdpa instead."
            )

if flash_attn_installed:
    print("flash_attention_2 available.")
else:
    print("Using sdpa attention (built into transformers, no install needed).")

# Imports
from tqdm import tqdm
from datasets import load_dataset
from src.llms import get_registed_model
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder
from src.trie import MarisaTrie
from src.graph_constrained_decoding import GraphConstrainedDecoding
from src.utils.qa_utils import eval_path_result_w_ans
from approach3_symbolic.type_oracle import TypeOracle
import src.utils as utils
import networkx as nx

print("All imports verified.")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB  |  VRAM: 42.4 GB  |  SM: 80
Cloning into '/content/graph-constrained-reasoning'...
remote: Enumerating objects: 377, done.
remote: Counting objects: 100% (377/377), done.
remote: Compressing objects: 100% (265/265), done.
remote: Total 377 (delta 162), reused 315 (delta 108), pack-reused 0 (from 0)
Receiving objects: 100% (377/377), 6.17 MiB | 29.80 MiB/s, done.
Resolving deltas: 100% (162/162), done.
Installing dependencies...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Using sdpa attention (built into transformers, no install needed).
All imports verified.


---
## 1. Configuration

In [3]:
#@title 1. Configuration

# Model — KG-specialized LLM
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"  #@param {type:"string"}

# Dataset
DATA_PATH = "rmanluo"         #@param {type:"string"}
DATASET = "RoG-webqsp"        #@param ["RoG-webqsp", "RoG-cwq"]
SPLIT = "test"                #@param ["test", "validation"]

# Decoding
INDEX_LEN = 2                 #@param {type:"integer"}
K = 10                        #@param {type:"integer"}
GEN_MODE = "group-beam"       #@param ["greedy", "group-beam", "beam"]
PROMPT_MODE = "zero-shot"     #@param ["zero-shot", "few-shot"]
MAX_NEW_TOKENS = 256          #@param {type:"integer"}

# Sampling
MAX_SAMPLES = 1628             #@param {type:"integer"}

# Output
FORCE_RERUN = False           #@param {type:"boolean"}

# Derived
ATTN_IMPL = "flash_attention_2" if (has_a100 and flash_attn_installed) else "sdpa"
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")

if IN_COLAB:
    OUTPUT_DIR = os.path.join(DRIVE_BASE, TIMESTAMP)
else:
    OUTPUT_DIR = os.path.join("results", "type_oracle_experiment", TIMESTAMP)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Model:       {MODEL_PATH}")
print(f"Dataset:     {DATA_PATH}/{DATASET} ({SPLIT})")
print(f"Max samples: {MAX_SAMPLES}")
print(f"Beam width:  {K}")
print(f"Index len:   {INDEX_LEN} hops")
print(f"Generation:  {GEN_MODE}")
print(f"Attention:   {ATTN_IMPL}")
print(f"GPU:         {gpu_name}")
print(f"Output:      {OUTPUT_DIR}")
print("=" * 60)

config = {
    "model_path": MODEL_PATH, "data_path": DATA_PATH,
    "dataset": DATASET, "split": SPLIT, "index_len": INDEX_LEN,
    "k": K, "gen_mode": GEN_MODE, "prompt_mode": PROMPT_MODE,
    "max_new_tokens": MAX_NEW_TOKENS, "max_samples": MAX_SAMPLES,
    "attn_impl": ATTN_IMPL, "gpu": gpu_name, "timestamp": TIMESTAMP,
}
with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

CONFIGURATION
Model:       rmanluo/GCR-Meta-Llama-3.1-8B-Instruct
Dataset:     rmanluo/RoG-webqsp (test)
Max samples: 1628
Beam width:  10
Index len:   2 hops
Generation:  group-beam
Attention:   sdpa
GPU:         NVIDIA A100-SXM4-40GB
Output:      /content/drive/MyDrive/GCR_TypeOracle/20260707_123532


---
## 2. Load Dataset

In [4]:
#@title 2. Load Dataset

dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
print(f"Full test set: {len(dataset)} questions")

if MAX_SAMPLES is not None and MAX_SAMPLES < len(dataset):
    dataset = dataset.select(range(MAX_SAMPLES))
    print(f"Subsampled to {len(dataset)} questions")

# Preview
print("\n--- Preview ---")
for i in range(min(3, len(dataset))):
    d = dataset[i]
    g = utils.build_graph(d["graph"])
    print(f"\n[{i}] Q: {d['question']}")
    print(f"    A: {d['answer']}")
    print(f"    Graph: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")

README.md:   0%|          | 0.00/900 [00:00<?, ?B/s]

data/train-00000-of-00002-d810a36ed97bc2(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

data/train-00001-of-00002-e53244e71082a3(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

data/validation-00000-of-00001-6ee6adc5b(…):   0%|          | 0.00/24.3M [00:00<?, ?B/s]

data/test-00000-of-00002-9ee8d68f7d951e1(…):   0%|          | 0.00/90.9M [00:00<?, ?B/s]

data/test-00001-of-00002-773a7b8213e159f(…):   0%|          | 0.00/93.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2826 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/246 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1628 [00:00<?, ? examples/s]

Full test set: 1628 questions

--- Preview ---

[0] Q: what does jamaican people speak
    A: ['Jamaican English', 'Jamaican Creole English Language']
    Graph: 1998 nodes, 5276 edges

[1] Q: what did james k polk do before he was president
    A: ['United States Representative', 'Governor of Tennessee', 'Speaker of the United States House of Representatives']
    Graph: 1959 nodes, 4240 edges

[2] Q: who plays ken barlow in coronation street
    A: ['William Roache']
    Graph: 1550 nodes, 2083 edges


---
## 3. TypeOracle SIR/FNR Evaluation (CPU-only)

Measures how many paths the TypeOracle prunes and whether any gold paths are lost.

- **SIR** = fraction of all paths pruned
- **FNR** = fraction of gold paths incorrectly pruned

In [ ]:
#@title 3. SIR/FNR Evaluation

total_before = 0
total_after = 0
total_range_blocked = 0
total_type_blocked = 0
n_range_fn = 0
n_type_fn = 0
total_gold = 0
skipped = 0

per_question_paths = []  # (qid, n_all, n_filtered) for later use

t0 = time.time()

for i, d in enumerate(dataset):
    try:
        oracle = TypeOracle.from_graph(d["graph"])
        ans_types = oracle.infer_answer_types(d["question"])
        g = utils.build_graph(d["graph"], undirected=False)
        entities = d.get("q_entity", [])

        if entities:
            paths_list = utils.dfs(g, entities, INDEX_LEN)
            kept = []
            for p in paths_list:
                admit = True
                for _, rel, tail in p:
                    if not oracle.range_gate(rel, tail):
                        total_range_blocked += 1
                        admit = False
                        break
                if admit:
                    terminal = p[-1][2]
                    if not oracle.type_gate(terminal, ans_types, len(p), INDEX_LEN):
                        total_type_blocked += 1
                        admit = False
                if admit:
                    kept.append(p)
            total_before += len(paths_list)
            total_after += len(kept)
            per_question_paths.append((d["id"], len(paths_list), len(kept)))
        else:
            skipped += 1
            per_question_paths.append((d["id"], 0, 0))

        # FNR on gold paths
        truth_paths = utils.get_truth_paths(d["q_entity"], d["a_entity"], g)
        for p in truth_paths:
            if not p:
                continue
            total_gold += 1
            if any(not oracle.range_gate(rel, tail) for _, rel, tail in p):
                n_range_fn += 1
            if not oracle.type_gate(p[-1][2], ans_types, len(p), INDEX_LEN):
                n_type_fn += 1
    except Exception as e:
        print(f"  Skipping {i}: {e}")
        skipped += 1

    if (i + 1) % 25 == 0:
        print(f"  {i + 1}/{len(dataset)} ({time.time() - t0:.1f}s)")

elapsed_sir = time.time() - t0
pruned = total_before - total_after
sir = pruned / max(1, total_before)

sir_metrics = {
    "samples": len(dataset), "skipped": skipped,
    "total_paths_raw": total_before, "total_paths_filtered": total_after,
    "pruned": pruned, "sir": round(sir, 4),
    "sir_type": round(total_type_blocked / max(1, total_before), 4),
    "sir_traj": round(total_range_blocked / max(1, total_before), 4),
    "gold_paths": total_gold,
    "fnr_type": round(n_type_fn / max(1, total_gold), 4),
    "fnr_range": round(n_range_fn / max(1, total_gold), 4),
    "elapsed_s": round(elapsed_sir, 1),
}

print(f"\n{'=' * 60}")
print("SIR/FNR RESULTS")
print(f"{'=' * 60}")
print(f"Paths (raw):      {total_before}")
print(f"Paths (filtered): {total_after}")
print(f"Pruned:           {pruned} ({sir * 100:.1f}%)")
print(f"SIR_type:         {sir_metrics['sir_type']}")
print(f"SIR_traj:         {sir_metrics['sir_traj']}")
print(f"Gold paths:       {total_gold}")
print(f"Type gate FNR:    {sir_metrics['fnr_type']}")
print(f"Range gate FNR:   {sir_metrics['fnr_range']}")
print(f"Time:             {elapsed_sir:.1f}s")
print(f"{'=' * 60}")

with open(os.path.join(OUTPUT_DIR, "sir_fnr_metrics.json"), "w") as f:
    json.dump(sir_metrics, f, indent=2)

  25/1628 (4.7s)
  50/1628 (8.3s)
  75/1628 (12.2s)
  100/1628 (15.9s)
  125/1628 (21.2s)
  150/1628 (24.5s)
  175/1628 (30.9s)
  200/1628 (36.5s)
  225/1628 (43.9s)
  250/1628 (48.7s)
  275/1628 (57.7s)
  300/1628 (62.9s)
  325/1628 (65.9s)
  350/1628 (70.2s)
  375/1628 (76.0s)
  400/1628 (84.1s)
  425/1628 (89.7s)
  450/1628 (92.8s)
  475/1628 (97.5s)
  500/1628 (100.8s)
  525/1628 (106.5s)
  550/1628 (112.0s)
  575/1628 (115.8s)
  600/1628 (120.4s)
  625/1628 (124.6s)
  650/1628 (129.3s)


---
## 4. Load KG-Specialized LLM

In [ ]:
#@title 4. Load Model

import argparse

LLM = get_registed_model(MODEL_PATH)
parser = argparse.ArgumentParser()
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_PATH
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

print(f"Loading {MODEL_PATH}...")
t0 = time.time()
model = LLM(args)
model.prepare_for_inference()
model.generation_cfg.temperature = None
model.generation_cfg.top_p = None
model.generation_cfg.top_k = None
model.model.generation_config.temperature = None
model.model.generation_config.top_p = None
model.model.generation_config.top_k = None

load_time = time.time() - t0
n_params = sum(p.numel() for p in model.model.parameters()) / 1e6
vram = torch.cuda.memory_allocated(0) / 1e9 if torch.cuda.is_available() else 0

print(f"Loaded in {load_time:.1f}s")
print(f"Parameters: {n_params:.1f}M  |  VRAM: {vram:.1f} GB")
print(f"Tokenizer vocab: {len(model.tokenizer)} tokens")

---
## 5. Helper Functions

In [ ]:
#@title 5. Helpers

def build_filtered_trie(tokenizer, question_dict, index_len, oracle):
    """Build a MarisaTrie from TypeOracle-filtered paths."""
    g = utils.build_graph(question_dict["graph"], undirected=False)
    entities = question_dict.get("q_entity", [])
    if not entities:
        return None, [], []

    all_paths = utils.dfs(g, entities, index_len)
    ans_types = oracle.infer_answer_types(question_dict["question"])

    filtered = []
    for p in all_paths:
        admit = True
        for _, rel, tail in p:
            if not oracle.range_gate(rel, tail):
                admit = False
                break
        if admit:
            terminal = p[-1][2]
            if not oracle.type_gate(terminal, ans_types, len(p), index_len):
                admit = False
        if admit:
            filtered.append(p)

    filtered_str = [utils.path_to_string(p) for p in filtered]
    if not filtered_str:
        return None, all_paths, filtered

    # Wrap in <PATH>...</PATH> tokens (matching PathGenerationWithAnswerPromptBuilder)
    PATH_START = "<PATH>"
    PATH_END = "</PATH>"
    wrapped = [f"{PATH_START}{s}{PATH_END}" for s in filtered_str]

    tokenized = tokenizer(wrapped, padding=False, add_special_tokens=False).input_ids
    trie = MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)
    return trie, all_paths, filtered


def build_unfiltered_trie(tokenizer, question_dict, index_len):
    """Build a MarisaTrie from all DFS paths (no filtering)."""
    g = utils.build_graph(question_dict["graph"], undirected=False)
    entities = question_dict.get("q_entity", [])
    if not entities:
        return None, []

    all_paths = utils.dfs(g, entities, index_len)
    all_str = [utils.path_to_string(p) for p in all_paths]
    if not all_str:
        return None, all_paths

    PATH_START = "<PATH>"
    PATH_END = "</PATH>"
    wrapped = [f"{PATH_START}{s}{PATH_END}" for s in all_str]

    tokenized = tokenizer(wrapped, padding=False, add_special_tokens=False).input_ids
    trie = MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)
    return trie, all_paths


def run_constrained_decoding(model, input_builder, data, trie):
    """Run graph-constrained decoding for a single question."""
    input_query, ground_paths, _ = input_builder.process_input(data, return_tire=False)
    start_token_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_START_TOKEN)
    end_token_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_END_TOKEN)
    llm_input = model.prepare_model_prompt(input_query)
    prediction = model.generate_sentence(
        llm_input, trie,
        start_token_ids=start_token_ids,
        end_token_ids=end_token_ids,
        enable_constrained_by_default=False,
    )
    return prediction, ground_paths


print("Helpers defined.")

---
## 6. TypeOracle-Filtered Graph-Constrained Decoding

For each question:
1. Enumerate all KG paths via DFS
2. Apply TypeOracle gates to filter
3. Build trie from filtered paths only
4. Run GCR constrained decoding with the filtered trie

In [ ]:
#@title 6. TypeOracle-Filtered Decoding

filtered_predictions_path = os.path.join(OUTPUT_DIR, "predictions_filtered.jsonl")

# Resume support
processed_filtered = set()
if os.path.exists(filtered_predictions_path) and not FORCE_RERUN:
    with open(filtered_predictions_path) as f:
        for line in f:
            try:
                processed_filtered.add(json.loads(line)["id"])
            except (json.JSONDecodeError, KeyError):
                pass
    print(f"Resuming: {len(processed_filtered)} already done")

fout_f = open(filtered_predictions_path, "a" if processed_filtered else "w")

input_builder = PathGenerationWithAnswerPromptBuilder(
    model.tokenizer, PROMPT_MODE, index_path_length=INDEX_LEN
)

hits_filtered = 0
n_done = 0
n_empty = 0

print(f"\n{'=' * 60}")
print(f"TypeOracle-Filtered Decoding — {len(dataset)} questions")
print(f"{'=' * 60}\n")

t0 = time.time()

for i, d in enumerate(dataset):
    qid = d["id"]
    if qid in processed_filtered:
        continue

    oracle = TypeOracle.from_graph(d["graph"])
    trie, all_paths, filtered = build_filtered_trie(
        model.tokenizer, d, INDEX_LEN, oracle
    )

    if trie is None:
        result = {
            "id": qid, "question": d["question"],
            "prediction": [], "ground_truth": d["answer"],
            "ground_truth_paths": [],
            "n_paths_all": len(all_paths), "n_paths_filtered": 0,
            "mode": "typeoracle_filtered",
        }
        fout_f.write(json.dumps(result) + "\n")
        fout_f.flush()
        processed_filtered.add(qid)
        n_done += 1
        n_empty += 1
        continue

    try:
        prediction, ground_paths = run_constrained_decoding(
            model, input_builder, d, trie
        )
    except Exception as e:
        print(f"  [{i}] Error: {e}")
        prediction = None

    if prediction is None:
        prediction = []

    result = {
        "id": qid, "question": d["question"],
        "prediction": prediction, "ground_truth": d["answer"],
        "ground_truth_paths": ground_paths,
        "n_paths_all": len(all_paths), "n_paths_filtered": len(filtered),
        "mode": "typeoracle_filtered",
    }
    fout_f.write(json.dumps(result) + "\n")
    fout_f.flush()
    processed_filtered.add(qid)
    n_done += 1

    if n_done % 10 == 0:
        elapsed = time.time() - t0
        rate = n_done / elapsed if elapsed > 0 else 0
        print(f"  [{n_done}/{len(dataset)}] {rate:.2f} q/s | {elapsed:.0f}s")

fout_f.close()
elapsed_filtered = time.time() - t0
print(f"\nDone: {n_done} questions ({n_empty} empty) in {elapsed_filtered:.1f}s")

---
## 7. Unfiltered GCR Baseline

Same pipeline but uses ALL DFS paths (no TypeOracle filtering).

In [ ]:
#@title 7. Unfiltered Baseline Decoding

unfiltered_predictions_path = os.path.join(OUTPUT_DIR, "predictions_unfiltered.jsonl")

processed_unfiltered = set()
if os.path.exists(unfiltered_predictions_path) and not FORCE_RERUN:
    with open(unfiltered_predictions_path) as f:
        for line in f:
            try:
                processed_unfiltered.add(json.loads(line)["id"])
            except (json.JSONDecodeError, KeyError):
                pass
    print(f"Resuming: {len(processed_unfiltered)} already done")

fout_u = open(unfiltered_predictions_path, "a" if processed_unfiltered else "w")

hits_unfiltered = 0
n_done_u = 0

print(f"\n{'=' * 60}")
print(f"Unfiltered Baseline Decoding — {len(dataset)} questions")
print(f"{'=' * 60}\n")

t0 = time.time()

for i, d in enumerate(dataset):
    qid = d["id"]
    if qid in processed_unfiltered:
        continue

    trie, all_paths = build_unfiltered_trie(model.tokenizer, d, INDEX_LEN)

    if trie is None:
        result = {
            "id": qid, "question": d["question"],
            "prediction": [], "ground_truth": d["answer"],
            "ground_truth_paths": [],
            "n_paths_all": 0, "mode": "unfiltered",
        }
        fout_u.write(json.dumps(result) + "\n")
        fout_u.flush()
        processed_unfiltered.add(qid)
        n_done_u += 1
        continue

    try:
        prediction, ground_paths = run_constrained_decoding(
            model, input_builder, d, trie
        )
    except Exception as e:
        print(f"  [{i}] Error: {e}")
        prediction = None

    if prediction is None:
        prediction = []

    result = {
        "id": qid, "question": d["question"],
        "prediction": prediction, "ground_truth": d["answer"],
        "ground_truth_paths": ground_paths,
        "n_paths_all": len(all_paths), "mode": "unfiltered",
    }
    fout_u.write(json.dumps(result) + "\n")
    fout_u.flush()
    processed_unfiltered.add(qid)
    n_done_u += 1

    if n_done_u % 10 == 0:
        elapsed = time.time() - t0
        rate = n_done_u / elapsed if elapsed > 0 else 0
        print(f"  [{n_done_u}/{len(dataset)}] {rate:.2f} q/s | {elapsed:.0f}s")

fout_u.close()
elapsed_unfiltered = time.time() - t0
print(f"\nDone: {n_done_u} questions in {elapsed_unfiltered:.1f}s")

---
## 8. Evaluate Both Runs

In [ ]:
#@title 8. Evaluation

print("\n=== TypeOracle-Filtered ===")
eval_path_result_w_ans(filtered_predictions_path)

print("\n=== Unfiltered Baseline ===")
eval_path_result_w_ans(unfiltered_predictions_path)

---
## 9. Head-to-Head Comparison

In [ ]:
#@title 9. Comparison

def load_preds(path):
    results = []
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    results.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return results


preds_f = load_preds(filtered_predictions_path)
preds_u = load_preds(unfiltered_predictions_path)

# Build lookup by id for paired comparison
preds_u_by_id = {p["id"]: p for p in preds_u}

n_f = len(preds_f)
n_u = len(preds_u)

# Compute hits using the official eval function's logic
from src.utils.qa_utils import eval_hit, normalize

def compute_hits(preds):
    hits = 0
    for p in preds:
        prediction = p.get("prediction", "")
        answer = list(set(p.get("ground_truth", [])))
        if isinstance(prediction, list):
            # Joint output: extract answers after "the answer is:"
            pred_str = " ".join(prediction)
        else:
            pred_str = prediction
        # Use the same matching as eval_path_result_w_ans
        from src.utils.qa_utils import extract_topk_prediction
        top_preds = extract_topk_prediction(pred_str, -1)
        pred_joined = " ".join(top_preds)
        for a in answer:
            if normalize(a) in normalize(pred_joined):
                hits += 1
                break
    return hits

hits_f = compute_hits(preds_f)
hits_u = compute_hits(preds_u)

# Path reduction
total_all = sum(p.get("n_paths_all", 0) for p in preds_f)
total_filtered = sum(p.get("n_paths_filtered", 0) for p in preds_f)
reduction = (1 - total_filtered / max(1, total_all)) * 100

comparison = {
    "filtered": {
        "n": n_f, "hits": hits_f,
        "hit_at_1": round(hits_f / max(1, n_f) * 100, 1),
        "avg_paths": round(total_filtered / max(1, n_f), 1),
        "time_s": round(elapsed_filtered, 1),
    },
    "unfiltered": {
        "n": n_u, "hits": hits_u,
        "hit_at_1": round(hits_u / max(1, n_u) * 100, 1),
        "avg_paths": round(total_all / max(1, n_u), 1),
        "time_s": round(elapsed_unfiltered, 1),
    },
    "path_reduction_pct": round(reduction, 1),
    "sir_fnr_metrics": sir_metrics,
}

print("=" * 60)
print("COMPARISON: TypeOracle-Filtered vs Unfiltered")
print("=" * 60)
print(f"\n{'Metric':<28} {'Filtered':<12} {'Unfiltered':<12}")
print("-" * 52)
print(f"{'Questions':<28} {n_f:<12} {n_u:<12}")
print(f"{'Hits@1':<28} {hits_f:<12} {hits_u:<12}")
print(f"{'Hit@1 (%)':<28} {comparison['filtered']['hit_at_1']:<12} {comparison['unfiltered']['hit_at_1']:<12}")
print(f"{'Avg paths/question':<28} {comparison['filtered']['avg_paths']:<12} {comparison['unfiltered']['avg_paths']:<12}")
print(f"{'Time (s)':<28} {comparison['filtered']['time_s']:<12} {comparison['unfiltered']['time_s']:<12}")
print(f"\nPath reduction: {reduction:.1f}%")

# Per-question comparison
print("\n--- Per-Question (first 20) ---")
print(f"{'Q#':<4} {'Hit(F)':<8} {'Hit(U)':<8} {'P_all':<7} {'P_filt':<8} Question")
print("-" * 80)
for i, pf in enumerate(preds_f[:20]):
    pu = preds_u_by_id.get(pf["id"], {})
    q_short = pf["question"][:45]
    hf = "Y" if pf.get("hit", False) else "N"
    hu = "Y" if pu.get("hit", False) else "N"
    p_all = pf.get("n_paths_all", "?")
    p_filt = pf.get("n_paths_filtered", "?")
    print(f"{i:<4} {hf:<8} {hu:<8} {p_all:<7} {p_filt:<8} {q_short}")
print(f"{'=' * 60}")

---
## 10. Save Results

In [ ]:
#@title 10. Save Everything

with open(os.path.join(OUTPUT_DIR, "comparison.json"), "w") as f:
    json.dump(comparison, f, indent=2)

report = (
    f"# TypeOracle + KG-LLM Experiment\n\n"
    f"**Timestamp:** {TIMESTAMP}\n"
    f"**Model:** {MODEL_PATH}\n"
    f"**Dataset:** {DATA_PATH}/{DATASET} ({SPLIT})\n"
    f"**Samples:** {MAX_SAMPLES}\n"
    f"**GPU:** {gpu_name}\n\n"
    f"## SIR/FNR\n\n"
    f"| Metric | Value |\n"
    f"|--------|-------|\n"
    f"| SIR | {sir_metrics['sir']} |\n"
    f"| SIR_type | {sir_metrics['sir_type']} |\n"
    f"| SIR_traj | {sir_metrics['sir_traj']} |\n"
    f"| FNR_type | {sir_metrics['fnr_type']} |\n"
    f"| FNR_range | {sir_metrics['fnr_range']} |\n\n"
    f"## Decoding\n\n"
    f"| Metric | Filtered | Unfiltered |\n"
    f"|--------|----------|------------|\n"
    f"| Hit@1 | {comparison['filtered']['hit_at_1']}% ({hits_f}/{n_f}) | "
    f"{comparison['unfiltered']['hit_at_1']}% ({hits_u}/{n_u}) |\n"
    f"| Avg paths/q | {comparison['filtered']['avg_paths']} | "
    f"{comparison['unfiltered']['avg_paths']} |\n"
    f"| Time | {comparison['filtered']['time_s']}s | "
    f"{comparison['unfiltered']['time_s']}s |\n\n"
    f"**Path reduction:** {reduction:.1f}%\n"
)
with open(os.path.join(OUTPUT_DIR, "report.md"), "w") as f:
    f.write(report)

print(f"Results saved to {OUTPUT_DIR}")
print("\nFiles:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f"  {fname:<40} {size:>8,} bytes")

In [ ]:
#@title 11. (Optional) Download as ZIP

if IN_COLAB:
    zip_name = f"TypeOracle_Experiment_{TIMESTAMP}.zip"
    !cd "{DRIVE_BASE}" && zip -r "/content/{zip_name}" "{TIMESTAMP}/"
    try:
        from google.colab import files
        files.download(f"/content/{zip_name}")
    except Exception:
        print(f"ZIP at /content/{zip_name}")